In [7]:
import pandas as pd
import seaborn as sns
import  matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,classification_report,confusion_matrix
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df=pd.read_csv("shop_smart_ecommerce.csv")

In [3]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [4]:
df.isnull().sum()

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [12]:
X=df.drop("Revenue",axis=1)
y=df["Revenue"].astype(int)

In [15]:
num_features = X.select_dtypes(["int64","float64"]).columns
cat_features = X.select_dtypes(["category","object"]).columns

In [17]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

In [32]:
transform=ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown="ignore"),cat_features),
        ("num",StandardScaler(),num_features)
    ]
)

In [38]:
dt=DecisionTreeClassifier(
    max_depth = 6,             #prevents overfitting
    min_samples_leaf=20,         #smoothing
    class_weight="balanced",      # prevents imbalance
    random_state=42
)

In [39]:
pipe=Pipeline(
    steps=[
        ("preprocess",transform),
        ("model",dt)
    ]
)

In [40]:
pipe.fit(X_train,y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [42]:
y_pred=pipe.predict(X_test)
print("f1: ",f1_score(y_test,y_pred))
print("confusion matrix: ",confusion_matrix(y_test,y_pred))
print("class. report: ",classification_report(y_test,y_pred))

f1:  0.6303270564915758
confusion matrix:  [[1775  309]
 [  64  318]]
class. report:                precision    recall  f1-score   support

           0       0.97      0.85      0.90      2084
           1       0.51      0.83      0.63       382

    accuracy                           0.85      2466
   macro avg       0.74      0.84      0.77      2466
weighted avg       0.89      0.85      0.86      2466



# Hyperparameter Tuning

In [43]:
from sklearn.model_selection import GridSearchCV

In [63]:
param_grid={
    "model__max_depth":[4,6,8],
    "model__min_samples_leaf":[20,30,40,50]
}

grid=GridSearchCV(
    pipe,
    param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1  #Use all available cores in GPU
)

In [64]:
grid.fit(X_train,y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [4, 6, ...], 'model__min_samples_leaf': [20, 30, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...), ('num', ...)]"


In [65]:
print("f1:",grid.best_score_)
print("best parameter:",grid.best_params_)

f1: 0.6343735129725652
best parameter: {'model__max_depth': 4, 'model__min_samples_leaf': 50}
